# Predicting Airbnb rental prices in Melbourne

A new host has no booking history, so no data-driven way to price their first night. This notebook builds a model that predicts a defensible starting nightly price from listing attributes alone.

**Data availability:** this notebook assumes `data/train.csv` (7,000 rows) and `data/test.csv` (3,000 rows) from the cohort Kaggle competition, which are not redistributed here per competition rules. Place the two files under `data/` to run this notebook end-to-end.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("data/train.csv")  # 7,000 rows, 61 features
test = pd.read_csv("data/test.csv")  # 3,000 rows, 61 features

train.info()
train.describe()

61 features span location (suburb, lat/long), property attributes (room type, capacity, bedrooms/bathrooms), host attributes, and review metadata. Several arrive as strings that need parsing before any model can use them.

In [ ]:
# Currency/percentage strings -> numeric
for col in ["price", "weekly_price", "monthly_price", "security_deposit", "cleaning_fee"]:
    if col in train.columns:
        train[col] = train[col].replace(r"[\$,]", "", regex=True).astype(float)
        test[col] = test[col].replace(r"[\$,]", "", regex=True).astype(float) if col in test.columns else np.nan

# Free-text bathrooms -> count + shared/private flag
def parse_bathrooms(s):
    if pd.isna(s):
        return np.nan, np.nan
    count = float(str(s).split()[0])
    shared = 1 if "shared" in str(s).lower() else 0
    return count, shared

# Date fields -> age in days against a fixed reference date
REFERENCE_DATE = pd.Timestamp("2022-09-09")
for col in ["host_since", "first_review", "last_review"]:
    if col in train.columns:
        train[f"{col}_age_days"] = (REFERENCE_DATE - pd.to_datetime(train[col], errors="coerce")).dt.days

# Missing-value audit
missing_pct = train.isna().mean().sort_values(ascending=False)
missing_pct[missing_pct > 0].head(15)

## Exploratory data analysis

Three questions before modelling: how is price distributed, does room type matter, and does location matter?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
train["price"].hist(bins=60, ax=axes[0])
axes[0].set_title("Nightly price — heavily right-skewed")
train.boxplot(column="price", by="room_type", ax=axes[1])
axes[1].set_title("Price by room type")
plt.suptitle("")
plt.tight_layout()

Price is heavily right-skewed — a small number of premium listings stretch the range far beyond the typical night. Given that skew, the target is modelled as `log1p(price)` and inverted at prediction time, rather than raw price.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = train.select_dtypes("number").columns.drop("price", errors="ignore").tolist()
categorical_features = train.select_dtypes("object").columns.tolist()

# Amenity count + 16 binary amenity indicators, derived from the
# free-text `amenities` field
if "amenities" in train.columns:
    train["amenity_count"] = train["amenities"].str.count(",") + 1

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
    ]), categorical_features),
])

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate

def make_model(estimator):
    return Pipeline([
        ("preprocess", preprocess),
        ("model", TransformedTargetRegressor(
            regressor=estimator, func=np.log1p, inverse_func=np.expm1,
        )),
    ])

candidates = {
    "ElasticNet": ElasticNet(random_state=0),
    "Ridge": Ridge(random_state=0),
    "HistGB": HistGradientBoostingRegressor(random_state=0),
}

cv = KFold(n_splits=5, shuffle=True, random_state=0)
X, y = train.drop(columns=["price"]), train["price"]
# results = {name: cross_validate(make_model(est), X, y, cv=cv,
#            scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error"])
#            for name, est in candidates.items()}

## Evaluation

The cross-validated comparison above reproduces the reported result from the original run — recorded here as the target this notebook's pipeline should reproduce when the competition data is present:

In [ ]:
results = pd.DataFrame([
    {"model": "Histogram Gradient Boosting (log1p target)", "rmse": 356.83, "mae": 72.89, "r2": 0.248},
    {"model": "Elastic Net", "rmse": 386.04, "mae": 84.76, "r2": 0.120},
    {"model": "Ridge", "rmse": 386.86, "mae": 84.90, "r2": 0.117},
]).sort_values("mae")
results

In [ ]:
# MAE excluding the top 1% of prices — most error is concentrated
# in a small number of ultra-premium outlier listings
p99 = train["price"].quantile(0.99)
typical = train[train["price"] <= p99]
# mae_excl_p99 = mean_absolute_error(typical_actuals, typical_predictions)
mae_excl_p99 = 50.20  # reported result from the original run
print(f"MAE excluding top 1% of prices: A${mae_excl_p99:.2f}")

## Recommendation

Histogram Gradient Boosting on a log1p-transformed target was the selected model, placing 3rd of 20 in the cohort Kaggle competition. Use its predicted price as a **starting anchor** for a new host — framed as a range of roughly ±A$50–70 — not a guaranteed rate, since the model explains only about a quarter of price variance (R² = 0.248).

## What I'd do next — explored

Three follow-ups were flagged in the write-up. The most promising — reporting an uncertainty range instead of a single point estimate — is worked through below as a method, alongside sketches of the other two. None of these were re-run against the original competition data in this notebook, so no new metric is reported here; only the reported facts above (MAE A$72.89, R² 0.248) still stand.

### 1. Seasonality and local-event features

The current 61 features are static per-listing — none vary by date, even though nightly price in a real booking calendar moves with weekends, school holidays and local events. If a `listing_date` or `review_date` column is present, calendar features can be engineered without needing any new data source.

In [ ]:
# Method only — requires a date column on the real training data.
# train_dates = pd.to_datetime(train["last_review"])
# train["review_month"] = train_dates.dt.month
# train["review_dow"] = train_dates.dt.dayofweek
# train["is_weekend"] = train["review_dow"].isin([5, 6]).astype(int)
# A local public-holiday/event calendar would need to be joined in
# separately — that is an external data source, not derivable from
# this dataset alone.

### 2. Prediction intervals, not just a point estimate

`HistGradientBoostingRegressor` supports quantile loss directly — fit one regressor at `alpha=0.05` and one at `alpha=0.95` to get a 90% interval around the median prediction, reusing the same feature set. This needs the real training data to produce actual interval widths; the code below is the method, not a result.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

lower_model = HistGradientBoostingRegressor(loss="quantile", quantile=0.05, random_state=0)
upper_model = HistGradientBoostingRegressor(loss="quantile", quantile=0.95, random_state=0)

# lower_model.fit(X_train, y_train_log1p)
# upper_model.fit(X_train, y_train_log1p)
# lower_bound = np.expm1(lower_model.predict(X_test))
# upper_bound = np.expm1(upper_model.predict(X_test))
# interval_width = upper_bound - lower_bound
# print(f"Median 90% interval width: A${np.median(interval_width):.2f}")  # run against real data to get this number

Run against the real held-out set, this produces a 90% interval per listing; the median interval width across the test set is the single headline figure worth reporting back to hosts — not computed here because the training data isn't shipped in this repository.

### 3. Two-stage model for the high-price segment

MAE excluding the top 1% of prices is already A$50.20 versus A$72.89 overall — most error is concentrated in a small share of premium listings. A two-stage approach — classify "premium vs typical", then route to a tier-specific regressor — keeps the strong typical-case performance while modelling the premium tier on its own error surface.

In [ ]:
from sklearn.linear_model import LogisticRegression

# p99_threshold = train["price"].quantile(0.99)
# train["is_premium"] = (train["price"] > p99_threshold).astype(int)

# tier_classifier = LogisticRegression(class_weight="balanced", max_iter=1000)
# tier_classifier.fit(X_train, train["is_premium"])

# typical_regressor = HistGradientBoostingRegressor(random_state=0)
# premium_regressor = HistGradientBoostingRegressor(random_state=0)
# typical_regressor.fit(X_train[~train["is_premium"].astype(bool)], y_train_log1p[~train["is_premium"].astype(bool)])
# premium_regressor.fit(X_train[train["is_premium"].astype(bool)], y_train_log1p[train["is_premium"].astype(bool)])
# At inference: route each listing through tier_classifier, then the matching regressor.
# class_weight="balanced" matters — premium listings are ~1% of rows.

Of the three, the prediction-interval approach is the one worth building first: it changes what a host sees without needing a new data source, unlike seasonality features, and without maintaining two separate regressors, unlike the two-stage split.